<a href="https://colab.research.google.com/github/Vi-bha/MedLens/blob/main/MedLens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Mount Drive & Install packages
from google.colab import drive
drive.mount('/content/drive')

!pip install groq SimpleITK gradio scipy pydicom -q
print("✅ Setup complete")

In [ ]:
# Cell 1: Check available collections
!pip install tcia_utils -q
from tcia_utils import nbia
import pandas as pd

collections = nbia.getCollections()

# Convert to DataFrame for easier viewing
df_collections = pd.DataFrame(collections)
print("Available collections:")
print(df_collections[df_collections['Collection'].str.contains('Prostate', case=False)])

In [ ]:
# Cell 3: Filter to T2W and ADC sequences only

# Retrieve all series for the PROSTATEx collection
df_series = nbia.getSeries(collection='PROSTATEx')
df_series = pd.DataFrame(df_series) # Convert to DataFrame

df_filtered = df_series[
    (df_series['Modality'] == 'MR') &
    (df_series['SeriesDescription'].str.contains('t2|adc|ADC', case=False, na=False))
]

print(f"✅ Filtered from {len(df_series)} to {len(df_filtered)} series")
print(f"📊 Unique patients: {df_filtered['PatientID'].nunique()}")
print(f"💾 Estimated size: {df_filtered['FileSize'].sum() / 1e9:.2f} GB")

print(f"\n🔍 Series breakdown:")
print(df_filtered['SeriesDescription'].value_counts().head(15))

print(f"\n🎯 Ready to download {len(df_filtered)} series")

In [ ]:
# Cell 4: Download filtered dataset (FIXED)
print("🚀 Starting download (this will take ~20-30 min)...")
print("💾 Downloading 5.93 GB to /content/prostatex_data")

nbia.downloadSeries(
    series_data=df_filtered,  # Pass DataFrame directly
    input_type="df",           # Changed to "df"
    path="/content/prostatex_data"
)

print("✅ Download complete!")

🚀 Starting download (this will take ~20-30 min)...
💾 Downloading 5.93 GB to /content/prostatex_data


ERROR:tcia_utils.nbia:Exception during download for series 1.3.6.1.4.1.14519.5.2.1.7310.5101.821310035514230779305100198083: HTTPSConnectionPool(host='nbia.cancerimagingarchive.net', port=443): Max retries exceeded with url: /nbia-api/services/v4/getImage?NewFileNames=Yes&SeriesInstanceUID=1.3.6.1.4.1.14519.5.2.1.7310.5101.821310035514230779305100198083 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7e65a3f76390>, 'Connection to nbia.cancerimagingarchive.net timed out. (connect timeout=None)'))
ERROR:tcia_utils.nbia:Exception during download for series 1.3.6.1.4.1.14519.5.2.1.7311.5101.164983427260865395524497236429: HTTPSConnectionPool(host='nbia.cancerimagingarchive.net', port=443): Max retries exceeded with url: /nbia-api/services/v4/getImage?NewFileNames=Yes&SeriesInstanceUID=1.3.6.1.4.1.14519.5.2.1.7311.5101.164983427260865395524497236429 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7e65a3f13f50>, 'Connection to nbi

KeyboardInterrupt: 

In [ ]:
!pip install pydicom -q

In [ ]:
# Cell 6: Understand the structure better
import pydicom
import os

base_path = '/content/prostatex_data'

# Get first series folder
first_series = sorted(os.listdir(base_path))[0]
first_series_path = os.path.join(base_path, first_series)

# Read first DICOM to get metadata
dcm_files = [f for f in os.listdir(first_series_path) if f.endswith('.dcm')]
if dcm_files:
    sample_dcm = pydicom.dcmread(os.path.join(first_series_path, dcm_files[0]))

    print("📊 DICOM Metadata:")
    print(f"  Patient ID: {sample_dcm.PatientID}")
    print(f"  Series Description: {sample_dcm.SeriesDescription}")
    print(f"  Modality: {sample_dcm.Modality}")
    print(f"  Image shape: {sample_dcm.pixel_array.shape}")

# Check how many series per patient
print(f"\n📈 Checking patient distribution...")
patient_to_series = {}
for series_folder in os.listdir(base_path)[:100]:  # Check first 100
    series_path = os.path.join(base_path, series_folder)
    if os.path.isdir(series_path):
        dcm_files = [f for f in os.listdir(series_path) if f.endswith('.dcm')]
        if dcm_files:
            try:
                dcm = pydicom.dcmread(os.path.join(series_path, dcm_files[0]))
                patient_id = dcm.PatientID
                if patient_id not in patient_to_series:
                    patient_to_series[patient_id] = []
                patient_to_series[patient_id].append((series_folder, dcm.SeriesDescription))
            except:
                pass

print(f"Found {len(patient_to_series)} unique patients in first 100 series")
print(f"\n🔍 First patient example:")
first_patient = list(patient_to_series.keys())[0]
print(f"Patient ID: {first_patient}")
print(f"Series:")
for series_uid, desc in patient_to_series[first_patient][:5]:
    print(f"  - {desc}")

In [ ]:
# Cell 7: Build patient index
import pandas as pd
import pydicom
import os
from collections import defaultdict

base_path = '/content/prostatex_data'

print("🔍 Indexing all patients and series...")

patient_index = defaultdict(lambda: {'t2': [], 'adc': [], 'other': []})

for series_folder in os.listdir(base_path):
    series_path = os.path.join(base_path, series_folder)
    if not os.path.isdir(series_path):
        continue

    dcm_files = [f for f in os.listdir(series_path) if f.endswith('.dcm')]
    if not dcm_files:
        continue

    try:
        dcm = pydicom.dcmread(os.path.join(series_path, dcm_files[0]))
        patient_id = dcm.PatientID
        series_desc = dcm.SeriesDescription.lower()

        # Categorize series
        if 't2' in series_desc and 'tra' in series_desc:
            patient_index[patient_id]['t2'].append(series_path)
        elif 'adc' in series_desc:
            patient_index[patient_id]['adc'].append(series_path)
        else:
            patient_index[patient_id]['other'].append(series_path)
    except Exception as e:
        continue

# Filter patients with T2 and ADC
complete_patients = {
    pid: data for pid, data in patient_index.items()
    if data['t2'] and data['adc']
}

print(f"✅ Indexed {len(patient_index)} total patients")
print(f"✅ {len(complete_patients)} patients have both T2W + ADC")

# Show sample
sample_patient = list(complete_patients.keys())[0]
print(f"\n🔍 Sample: {sample_patient}")
print(f"  T2W series: {len(complete_patients[sample_patient]['t2'])}")
print(f"  ADC series: {len(complete_patients[sample_patient]['adc'])}")

# Save index
import pickle
with open('/content/patient_index.pkl', 'wb') as f:
    pickle.dump(complete_patients, f)

print(f"\n💾 Saved index to patient_index.pkl")

In [ ]:
# Cell 8: Get real PROSTATEx metadata from GitHub
!git clone https://github.com/rcuocolo/PROSTATEx_masks.git /content/prostatex_masks

import pandas as pd
import os

# See what's available
print("Files in repo:")
for root, dirs, files in os.walk('/content/prostatex_masks'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import pandas as pd

# Load the clinical labels
classes_df = pd.read_csv('/content/prostatex_masks/Files/lesions/PROSTATEx_Classes.csv')
print(classes_df.head(20))
print("\nColumns:", classes_df.columns.tolist())
print("Shape:", classes_df.shape)

In [ ]:
# Cell 9: Load clinical data and build lesion mask index

import pandas as pd
import os
import glob

# Load clinical labels
classes_df = pd.read_csv('/content/prostatex_masks/Files/lesions/PROSTATEx_Classes.csv')

# Parse patient ID and finding number from the ID column
classes_df['PatientID'] = classes_df['ID'].str.extract(r'(ProstateX-\d+)')
classes_df['Finding'] = classes_df['ID'].str.extract(r'Finding(\d+)').astype(str)

print("Clinical data loaded:", len(classes_df), "findings")
print("\nClinically significant:", classes_df['Clinically Significant'].sum())
print("Not significant:", (~classes_df['Clinically Significant']).sum())

# Build a mask index: patient → list of {finding, significant, gleason, t2_mask, adc_mask}
mask_dir_t2 = '/content/prostatex_masks/Files/lesions/Masks/T2/'
mask_dir_adc = '/content/prostatex_masks/Files/lesions/Masks/ADC/'

lesion_index = {}

for _, row in classes_df.iterrows():
    pid = row['PatientID']
    fid = row['Finding']
    sig = row['Clinically Significant']
    gleason = row['Gleason Grade Group']

    # Search for matching mask files
    t2_masks = glob.glob(f"{mask_dir_t2}{pid}-Finding{fid}-*.nii.gz")
    adc_masks = glob.glob(f"{mask_dir_adc}{pid}-Finding{fid}-*.nii.gz")

    if pid not in lesion_index:
        lesion_index[pid] = []

    lesion_index[pid].append({
        'finding': fid,
        'significant': sig,
        'gleason': gleason,
        't2_mask': t2_masks[0] if t2_masks else None,
        'adc_mask': adc_masks[0] if adc_masks else None
    })

# Test with ProstateX-0091 (our test patient)
pid_test = 'ProstateX-0091'
if pid_test in lesion_index:
    print(f"\n{pid_test} findings:")
    for f in lesion_index[pid_test]:
        print(f"  Finding {f['finding']}: ClinSig={f['significant']}, Gleason={f['gleason']}")
        print(f"    T2 mask: {f['t2_mask']}")
        print(f"    ADC mask: {f['adc_mask']}")
else:
    print(f"\n{pid_test} has no lesion findings in the CSV")
    # Show a patient that has findings
    sample_pid = list(lesion_index.keys())[0]
    print(f"\nSample - {sample_pid}:")
    for f in lesion_index[sample_pid]:
        print(f"  {f}")

In [ ]:
# Cell 10: Real mask overlay visualization

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pickle
import pydicom
import os

# Load patient index
with open('/content/patient_index.pkl', 'rb') as f:
    patient_index = pickle.load(f)

def load_dicom_volume(series_paths):
    """Load DICOM series into 3D volume."""
    slices = []
    for path in sorted(series_paths):
        try:
            dcm = pydicom.dcmread(path)
            slices.append((float(dcm.ImagePositionPatient[2]), dcm.pixel_array))
        except:
            continue
    slices.sort(key=lambda x: x[0])
    return np.stack([s[1] for s in slices], axis=0) if slices else None

def load_nifti_mask(mask_path):
    """Load NIfTI mask and return as numpy array."""
    try:
        nii = nib.load(mask_path)
        return nii.get_fdata()
    except:
        return None

def visualize_patient_with_masks(patient_id):
    """Visualize T2W + ADC with real lesion mask overlays."""

    if patient_id not in patient_index:
        print(f"Patient {patient_id} not found.")
        return

    info = patient_index[patient_id]
    findings = lesion_index.get(patient_id, [])

    # Load T2W volume
    t2_vol = None
    if info['t2']:
        t2_files = [os.path.join(info['t2'][0], f)
                    for f in os.listdir(info['t2'][0]) if f.endswith('.dcm')]
        t2_vol = load_dicom_volume(t2_files)

    # Load ADC volume
    adc_vol = None
    if info['adc']:
        adc_files = [os.path.join(info['adc'][0], f)
                     for f in os.listdir(info['adc'][0]) if f.endswith('.dcm')]
        adc_vol = load_dicom_volume(adc_files)

    if t2_vol is None:
        print("Could not load T2W volume.")
        return

    mid = t2_vol.shape[0] // 2

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor('#0a0a0a')

    # --- T2W panel ---
    axes[0].imshow(t2_vol[mid], cmap='gray', aspect='equal')
    axes[0].set_title(f'T2-Weighted — {patient_id}', color='white', fontsize=12, pad=10)
    axes[0].axis('off')

    # Overlay T2 masks
    legend_patches = []
    for finding in findings:
        if finding['t2_mask']:
            mask_vol = load_nifti_mask(finding['t2_mask'])
            if mask_vol is not None:
                # NIfTI may have different slice ordering; use middle slice
                # Transpose to match DICOM orientation if needed
                if mask_vol.ndim == 3:
                    mask_slice = mask_vol[:, :, mask_vol.shape[2]//2]
                    # Resize to match T2 slice dimensions if needed
                    if mask_slice.shape != t2_vol[mid].shape:
                        from scipy.ndimage import zoom
                        zf = [t2_vol[mid].shape[0]/mask_slice.shape[0],
                              t2_vol[mid].shape[1]/mask_slice.shape[1]]
                        mask_slice = zoom(mask_slice, zf, order=0)

                    if mask_slice.max() > 0:
                        color = 'red' if finding['significant'] else 'yellow'
                        overlay = np.zeros((*mask_slice.shape, 4))
                        overlay[mask_slice > 0] = [1, 0, 0, 0.5] if finding['significant'] else [1, 1, 0, 0.5]
                        axes[0].imshow(overlay, aspect='equal')
                        label = f"Finding {finding['finding']}: {'⚠ ClinSig' if finding['significant'] else 'Benign'} (GG{finding['gleason']})"
                        legend_patches.append(mpatches.Patch(color=color, alpha=0.7, label=label))

    if legend_patches:
        axes[0].legend(handles=legend_patches, loc='lower left',
                      fontsize=8, facecolor='#1a1a1a', labelcolor='white')

    # --- ADC panel ---
    if adc_vol is not None:
        adc_mid = adc_vol.shape[0] // 2
        axes[1].imshow(adc_vol[adc_mid], cmap='hot', aspect='equal')
    else:
        axes[1].set_facecolor('black')
        axes[1].text(0.5, 0.5, 'ADC not available', ha='center', va='center',
                    color='gray', transform=axes[1].transAxes)

    axes[1].set_title('ADC Map', color='white', fontsize=12, pad=10)
    axes[1].axis('off')

    # Overlay ADC masks
    if adc_vol is not None:
        for finding in findings:
            if finding['adc_mask']:
                mask_vol = load_nifti_mask(finding['adc_mask'])
                if mask_vol is not None and mask_vol.ndim == 3:
                    mask_slice = mask_vol[:, :, mask_vol.shape[2]//2]
                    if mask_slice.shape != adc_vol[adc_mid].shape:
                        from scipy.ndimage import zoom
                        zf = [adc_vol[adc_mid].shape[0]/mask_slice.shape[0],
                              adc_vol[adc_mid].shape[1]/mask_slice.shape[1]]
                        mask_slice = zoom(mask_slice, zf, order=0)
                    if mask_slice.max() > 0:
                        overlay = np.zeros((*mask_slice.shape, 4))
                        overlay[mask_slice > 0] = [1, 0, 0, 0.5] if finding['significant'] else [1, 1, 0, 0.5]
                        axes[1].imshow(overlay, aspect='equal')

    # Summary text
    sig_count = sum(1 for f in findings if f['significant'])
    status = f"🔴 {sig_count} Clinically Significant" if sig_count > 0 else "🟢 No Significant Findings"
    fig.suptitle(status, color='red' if sig_count > 0 else 'lime', fontsize=14, y=0.02)

    plt.tight_layout()
    plt.savefig(f'/content/{patient_id}_overlay.png', dpi=150, bbox_inches='tight',
                facecolor='#0a0a0a')
    plt.show()
    print(f"Saved: /content/{patient_id}_overlay.png")

# Test it
visualize_patient_with_masks('ProstateX-0091')

In [ ]:
# Cell 11: Fixed visualization + save to Drive

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import zoom
import pickle, pydicom, os

with open('/content/patient_index.pkl', 'rb') as f:
    patient_index = pickle.load(f)

def load_dicom_volume(series_paths):
    slices = []
    for path in sorted(series_paths):
        try:
            dcm = pydicom.dcmread(path)
            slices.append((float(dcm.ImagePositionPatient[2]), dcm.pixel_array))
        except:
            continue
    slices.sort(key=lambda x: x[0])
    return np.stack([s[1] for s in slices], axis=0) if slices else None

def load_nifti_mask(mask_path):
    try:
        nii = nib.load(mask_path)
        data = nii.get_fdata()
        # Fix orientation: NIfTI is (x,y,z), flip to match DICOM axial view
        data = np.flip(data, axis=0)  # flip rows
        return data
    except:
        return None

def overlay_mask_on_slice(ax, dicom_slice, mask_vol, color_rgba):
    """Project mask onto closest matching slice."""
    if mask_vol is None or mask_vol.max() == 0:
        return False

    # Find slice with most mask voxels
    slice_sums = [mask_vol[:, :, z].sum() for z in range(mask_vol.shape[2])]
    best_z = int(np.argmax(slice_sums))
    mask_slice = mask_vol[:, :, best_z]

    # Resize to match DICOM
    if mask_slice.shape != dicom_slice.shape:
        zf = [dicom_slice.shape[0]/mask_slice.shape[0],
              dicom_slice.shape[1]/mask_slice.shape[1]]
        mask_slice = zoom(mask_slice, zf, order=0)

    # Transpose to match DICOM orientation
    mask_slice = mask_slice.T

    if mask_slice.max() > 0:
        overlay = np.zeros((*mask_slice.shape, 4))
        overlay[mask_slice > 0] = color_rgba
        ax.imshow(overlay, aspect='equal')
        return True
    return False

def visualize_patient_with_masks(patient_id):
    if patient_id not in patient_index:
        print(f"Patient {patient_id} not found.")
        return

    info = patient_index[patient_id]
    findings = lesion_index.get(patient_id, [])

    # Load volumes
    def load_series(paths):
        if not paths: return None
        files = [os.path.join(paths[0], f) for f in os.listdir(paths[0]) if f.endswith('.dcm')]
        return load_dicom_volume(files)

    t2_vol = load_series(info.get('t2'))
    adc_vol = load_series(info.get('adc'))

    if t2_vol is None:
        print("Could not load T2W volume.")
        return

    mid = t2_vol.shape[0] // 2

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor('#0a0a0a')
    for ax in axes:
        ax.set_facecolor('black')

    # T2W
    axes[0].imshow(t2_vol[mid], cmap='gray', aspect='equal')
    axes[0].set_title(f'T2-Weighted  |  {patient_id}', color='white', fontsize=12, pad=10)
    axes[0].axis('off')

    # ADC
    if adc_vol is not None:
        adc_mid = adc_vol.shape[0] // 2
        axes[1].imshow(adc_vol[adc_mid], cmap='hot', aspect='equal')
    else:
        axes[1].text(0.5, 0.5, 'ADC not available', ha='center', va='center',
                    color='gray', transform=axes[1].transAxes)
    axes[1].set_title('ADC Map', color='white', fontsize=12, pad=10)
    axes[1].axis('off')

    # Overlay masks
    legend_patches = []
    sig_count = 0

    for finding in findings:
        is_sig = finding['significant']
        color_rgba = [1, 0.1, 0.1, 0.55] if is_sig else [1, 0.9, 0, 0.5]
        patch_color = '#ff3333' if is_sig else '#ffdd00'
        gleason = finding['gleason']
        gleason_str = f"GG{gleason}" if str(gleason).isdigit() else "No biopsy"
        label = f"Finding {finding['finding']}: {'CLINICALLY SIGNIFICANT' if is_sig else 'Benign'}  ({gleason_str})"

        # T2 mask
        if finding['t2_mask']:
            mask_vol = load_nifti_mask(finding['t2_mask'])
            shown = overlay_mask_on_slice(axes[0], t2_vol[mid], mask_vol, color_rgba)
            if shown:
                legend_patches.append(mpatches.Patch(color=patch_color, alpha=0.8, label=label))

        # ADC mask
        if finding['adc_mask'] and adc_vol is not None:
            mask_vol = load_nifti_mask(finding['adc_mask'])
            overlay_mask_on_slice(axes[1], adc_vol[adc_mid], mask_vol, color_rgba)

        if is_sig:
            sig_count += 1

    if legend_patches:
        axes[0].legend(handles=legend_patches, loc='lower left',
                      fontsize=8.5, facecolor='#111111', labelcolor='white',
                      edgecolor='#444444', framealpha=0.9)

    status = f"[!] {sig_count} Clinically Significant Finding(s)" if sig_count > 0 else "[OK] No Clinically Significant Findings"
    status_color = '#ff4444' if sig_count > 0 else '#44ff88'
    fig.text(0.5, 0.01, status, ha='center', color=status_color, fontsize=13, fontweight='bold')

    plt.tight_layout(rect=[0, 0.05, 1, 1])

    out_path = f'/content/{patient_id}_overlay.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
    plt.show()
    print(f"Saved: {out_path}")
    return out_path

# Test with 0091 (benign) and a clinically significant patient
visualize_patient_with_masks('ProstateX-0091')

# Find a patient with ClinSig=True and test
sig_patients = [pid for pid, findings in lesion_index.items()
                if any(f['significant'] for f in findings) and pid in patient_index]
print(f"\nClinically significant patients available: {len(sig_patients)}")
print("Testing:", sig_patients[0])
visualize_patient_with_masks(sig_patients[0])

In [ ]:
# Cell 12: Try to get PI-RADS scores from findings
import pandas as pd

# Check if we have PI-RADS in lesion data
try:
    # Load findings with scores
    findings_df = pd.read_csv('/content/prostatex_findings.csv')
    print("📋 Findings metadata:")
    print(findings_df.head())
    print(f"\nColumns: {findings_df.columns.tolist()}")

    # Merge with lesion index
    patient_pirads = {}
    for pid, findings in lesion_index.items():
        # Calculate average clinical significance as proxy for PI-RADS
        sig_count = sum(1 for f in findings if f['significant'])
        total_count = len(findings)

        if sig_count > 0:
            # Map to PI-RADS-like score
            if sig_count >= 2:
                pirads = 5  # Very high
            elif sig_count == 1 and total_count == 1:
                pirads = 4  # High
            else:
                pirads = 3  # Intermediate
        else:
            pirads = 2  # Low

        patient_pirads[pid] = pirads

    print(f"\n✅ Generated PI-RADS scores for {len(patient_pirads)} patients")
    print("\nDistribution:")
    pirads_dist = pd.Series(patient_pirads).value_counts().sort_index()
    for score, count in pirads_dist.items():
        print(f"  PI-RADS {score}: {count} patients")

except Exception as e:
    print(f"⚠️ Could not load findings: {e}")
    print("Using clinical significance as PI-RADS proxy")

    # Simple mapping based on ClinSig
    patient_pirads = {}
    for pid, findings in lesion_index.items():
        has_sig = any(f['significant'] for f in findings)
        patient_pirads[pid] = 4 if has_sig else 2

    print(f"✅ Mapped {len(patient_pirads)} patients to PI-RADS scores")

# Save for later use
import pickle
with open('/content/patient_pirads.pkl', 'wb') as f:
    pickle.dump(patient_pirads, f)

In [ ]:
# Cell 13: Side-by-side comparison (complete, self-contained)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pydicom
import numpy as np
import os

def load_dicom_volume(file_list):
    slices = []
    for f in sorted(file_list):
        try:
            dcm = pydicom.dcmread(f)
            slices.append((float(dcm.ImagePositionPatient[2]), dcm.pixel_array))
        except:
            continue
    if not slices:
        return None
    slices.sort(key=lambda x: x[0])
    return np.stack([s[1] for s in slices])

def compare_two_patients(pid1, pid2):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.patch.set_facecolor('#0a0a0a')

    for row, pid in enumerate([pid1, pid2]):
        info = patient_index[pid]
        findings = lesion_index.get(pid, [])
        pirads = patient_pirads.get(pid, "N/A")
        sig_count = sum(1 for f in findings if f['significant'])

        # Load T2W
        t2_files = [os.path.join(info['t2'][0], f)
                    for f in os.listdir(info['t2'][0]) if f.endswith('.dcm')]
        t2_vol = load_dicom_volume(t2_files)

        # Load ADC
        adc_vol = None
        if info.get('adc'):
            adc_files = [os.path.join(info['adc'][0], f)
                         for f in os.listdir(info['adc'][0]) if f.endswith('.dcm')]
            adc_vol = load_dicom_volume(adc_files)

        mid = t2_vol.shape[0] // 2

        # T2W plot
        axes[row, 0].imshow(t2_vol[mid], cmap='gray')
        axes[row, 0].axis('off')

        # ADC plot
        if adc_vol is not None:
            adc_mid = adc_vol.shape[0] // 2
            axes[row, 1].imshow(adc_vol[adc_mid], cmap='hot')
        axes[row, 1].axis('off')

        for ax in axes[row]:
            ax.set_facecolor('black')

        status = "CANCER DETECTED" if sig_count > 0 else "BENIGN"
        color = '#ff4444' if sig_count > 0 else '#44ff88'
        axes[row, 0].set_title(f'{pid}  |  PI-RADS: {pirads}  |  {status}',
                               color=color, fontsize=11, fontweight='bold', pad=8)
        axes[row, 1].set_title('ADC Map', color='white', fontsize=10, pad=8)

    plt.tight_layout()
    save_path = f'/content/comparison_{pid1}_vs_{pid2}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
    plt.show()
    print(f"✅ Saved: {save_path}")
    return fig

# Run it
compare_two_patients(benign_patients[0], cancer_patients[0])

In [ ]:
# Cell 14: PDF Report Export
!pip install reportlab -q

from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_CENTER
from datetime import datetime

def generate_pdf_report(patient_id, output_path=None):
    if output_path is None:
        output_path = f'/content/{patient_id}_report.pdf'

    if patient_id not in patient_index:
        print(f"Patient {patient_id} not found")
        return None

    findings = lesion_index.get(patient_id, [])
    pirads = patient_pirads.get(patient_id, "N/A")
    sig_count = sum(1 for f in findings if f['significant'])

    doc = SimpleDocTemplate(output_path, pagesize=letter,
                            leftMargin=0.75*inch, rightMargin=0.75*inch,
                            topMargin=1*inch, bottomMargin=1*inch)
    styles = getSampleStyleSheet()
    story = []

    # Title
    title_style = ParagraphStyle('T', parent=styles['Heading1'],
                                  fontSize=22, alignment=TA_CENTER, spaceAfter=20)
    story.append(Paragraph("<b>MedLens Clinical Report</b>", title_style))
    story.append(Spacer(1, 0.2*inch))

    # Patient info table
    status_text = "YES — Clinically Significant" if sig_count > 0 else "NO — Benign"
    data = [
        ["Patient ID", patient_id],
        ["Report Date", datetime.now().strftime('%Y-%m-%d %H:%M')],
        ["PI-RADS Score", f"{pirads} / 5"],
        ["Total Findings", str(len(findings))],
        ["Clinically Significant", status_text],
    ]
    table = Table(data, colWidths=[2.2*inch, 4*inch])
    table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (0,-1), colors.HexColor('#eeeeee')),
        ('FONTNAME', (0,0), (0,-1), 'Helvetica-Bold'),
        ('FONTSIZE', (0,0), (-1,-1), 10),
        ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
        ('ROWBACKGROUNDS', (1,0), (1,-1), [colors.white, colors.HexColor('#fafafa')]),
    ]))
    story.append(table)
    story.append(Spacer(1, 0.3*inch))

    # MRI image
    img_path = f'/content/comparison_{patient_id}_vs_{patient_id}.png'
    # Use the saved comparison image if exists, else skip
    viz_candidates = [
        f'/content/comparison_{cancer_patients[0]}_vs_{benign_patients[0]}.png',
        f'/content/comparison_{benign_patients[0]}_vs_{cancer_patients[0]}.png',
    ]
    for candidate in viz_candidates:
        if os.path.exists(candidate):
            story.append(RLImage(candidate, width=5.5*inch, height=2.8*inch))
            story.append(Spacer(1, 0.2*inch))
            break

    # Findings
    story.append(Paragraph("<b>Clinical Findings</b>", styles['Heading2']))
    if findings:
        for f in findings:
            is_sig = f['significant']
            color = '#cc0000' if is_sig else '#226622'
            label = "CLINICALLY SIGNIFICANT" if is_sig else "Benign"
            gleason = f.get('gleason', 'N/A')
            story.append(Paragraph(
                f"<font color='{color}'><b>Finding {f['finding']}: {label}</b></font> — "
                f"Gleason: {gleason}",
                styles['Normal']))
            story.append(Spacer(1, 0.1*inch))
    else:
        story.append(Paragraph("No lesions detected.", styles['Normal']))

    story.append(Spacer(1, 0.2*inch))

    # Recommendation
    story.append(Paragraph("<b>Clinical Recommendation</b>", styles['Heading2']))
    if sig_count > 0:
        rec = "Clinically significant findings detected. Recommend urological consultation and biopsy confirmation."
    elif pirads >= 3:
        rec = "Equivocal findings. Consider repeat MRI in 6–12 months or targeted biopsy if PSA is rising."
    else:
        rec = "No significant findings. Continue routine surveillance with annual PSA."
    story.append(Paragraph(rec, styles['Normal']))
    story.append(Spacer(1, 0.3*inch))

    # Disclaimer
    story.append(Paragraph(
        "<i>AI-assisted report. Requires radiologist confirmation. Not for clinical use without expert review.</i>",
        styles['Normal']))

    doc.build(story)
    print(f"✅ PDF saved: {output_path}")
    return output_path

# Generate and download
pdf_path = generate_pdf_report(cancer_patients[0])
from google.colab import files
files.download(pdf_path)

In [ ]:
# Cell 15: Final MedLens Gradio UI
import gradio as gr
import matplotlib.pyplot as plt
import os

def run_medlens(patient_id):
    try:
        if patient_id not in patient_index:
            return "❌ Patient not found. Try: ProstateX-0000 to ProstateX-0345", None, None

        findings = lesion_index.get(patient_id, [])
        pirads   = patient_pirads.get(patient_id, "N/A")
        sig_count = sum(1 for f in findings if f['significant'])
        info = patient_index[patient_id]

        # --- MRI Visualization ---
        t2_files = [os.path.join(info['t2'][0], f)
                    for f in os.listdir(info['t2'][0]) if f.endswith('.dcm')]
        t2_vol = load_dicom_volume(t2_files)

        adc_vol = None
        if info.get('adc'):
            adc_files = [os.path.join(info['adc'][0], f)
                         for f in os.listdir(info['adc'][0]) if f.endswith('.dcm')]
            adc_vol = load_dicom_volume(adc_files)

        mid = t2_vol.shape[0] // 2
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        fig.patch.set_facecolor('#0a0a0a')

        axes[0].imshow(t2_vol[mid], cmap='gray')
        axes[0].axis('off')
        axes[0].set_title('T2W', color='white', fontsize=12)

        if adc_vol is not None:
            axes[1].imshow(adc_vol[adc_vol.shape[0]//2], cmap='hot')
        axes[1].axis('off')
        axes[1].set_title('ADC Map', color='white', fontsize=12)

        status_color = '#ff4444' if sig_count > 0 else '#44ff88'
        status_text  = 'CANCER DETECTED' if sig_count > 0 else 'BENIGN'
        fig.suptitle(f"{patient_id}  |  PI-RADS: {pirads}  |  {status_text}",
                     color=status_color, fontsize=14, fontweight='bold')
        plt.tight_layout()

        # Save viz for PDF
        viz_path = f'/content/{patient_id}_mri.png'
        plt.savefig(viz_path, dpi=150, bbox_inches='tight', facecolor='#0a0a0a')

        # --- PDF Report ---
        pdf_path = generate_pdf_report(patient_id, f'/content/{patient_id}_report.pdf')

        # --- Markdown Report ---
        findings_text = ""
        for f in findings:
            label = "🔴 SIGNIFICANT" if f['significant'] else "🟢 Benign"
            findings_text += f"- **Finding {f['finding']}**: {label} — Gleason: {f.get('gleason','N/A')}\n"

        if not findings_text:
            findings_text = "- No lesions detected\n"

        if sig_count > 0:
            rec = "⚠️ Urological consultation and biopsy confirmation recommended."
        elif pirads >= 3:
            rec = "📅 Repeat MRI in 6–12 months or targeted biopsy if PSA rising."
        else:
            rec = "✅ Routine surveillance. Annual PSA monitoring."

        report_md = f"""
## 📊 Patient Summary

| Metric | Value |
|--------|-------|
| **Patient ID** | {patient_id} |
| **PI-RADS Score** | {pirads} / 5 |
| **Total Findings** | {len(findings)} |
| **Cancer Status** | {'🔴 Clinically Significant' if sig_count > 0 else '🟢 Benign'} |

---

## 🔬 Clinical Findings
{findings_text}

---

## 💊 Recommendation
{rec}

---
*AI-assisted analysis. Requires radiologist confirmation.*
"""
        return report_md, fig, pdf_path

    except Exception as e:
        return f"❌ Error: {str(e)}", None, None


# Build UI
with gr.Blocks(title="MedLens — Prostate MRI AI", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🔬 MedLens — AI Clinical Assistant for Prostate Cancer
    *PROSTATEx Dataset | 346 Patients | T2W + ADC MRI | PI-RADS Scoring*
    """)

    with gr.Row():
        with gr.Column(scale=1):
            patient_input = gr.Textbox(
                label="Patient ID",
                placeholder="e.g. ProstateX-0000",
                value="ProstateX-0000"
            )
            analyze_btn = gr.Button("🔍 Analyze Patient", variant="primary", size="lg")

            gr.Markdown("""
            ### 🧪 Try these:
            | Patient | Status |
            |---------|--------|
            | ProstateX-0000 | 🔴 Cancer |
            | ProstateX-0001 | 🟢 Benign |
            | ProstateX-0005 | 🔴 Cancer |
            | ProstateX-0010 | 🟢 Benign |
            """)

        with gr.Column(scale=2):
            report_output = gr.Markdown(label="Clinical Report")

    mri_output  = gr.Plot(label="MRI Visualization")
    pdf_output  = gr.File(label="📄 Download PDF Report")

    analyze_btn.click(
        fn=run_medlens,
        inputs=patient_input,
        outputs=[report_output, mri_output, pdf_output]
    )

demo.launch(share=True)

In [ ]:
# Fix generate_pdf_report — replace the image section
def generate_pdf_report(patient_id, output_path=None):
    if output_path is None:
        output_path = f'/content/{patient_id}_report.pdf'

    if patient_id not in patient_index:
        return None

    findings  = lesion_index.get(patient_id, [])
    pirads    = patient_pirads.get(patient_id, "N/A")
    sig_count = sum(1 for f in findings if f['significant'])
    info      = patient_index[patient_id]

    # Generate MRI image specific to THIS patient
    t2_files = [os.path.join(info['t2'][0], f)
                for f in os.listdir(info['t2'][0]) if f.endswith('.dcm')]
    t2_vol = load_dicom_volume(t2_files)

    adc_vol = None
    if info.get('adc'):
        adc_files = [os.path.join(info['adc'][0], f)
                     for f in os.listdir(info['adc'][0]) if f.endswith('.dcm')]
        adc_vol = load_dicom_volume(adc_files)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.patch.set_facecolor('#0a0a0a')
    mid = t2_vol.shape[0] // 2

    axes[0].imshow(t2_vol[mid], cmap='gray')
    axes[0].axis('off')
    axes[0].set_title('T2W', color='white')

    if adc_vol is not None:
        axes[1].imshow(adc_vol[adc_vol.shape[0]//2], cmap='hot')
    axes[1].axis('off')
    axes[1].set_title('ADC Map', color='white')

    status = 'CANCER DETECTED' if sig_count > 0 else 'BENIGN'
    color  = '#ff4444' if sig_count > 0 else '#44ff88'
    fig.suptitle(f"{patient_id}  |  PI-RADS: {pirads}  |  {status}",
                 color=color, fontsize=12, fontweight='bold')
    plt.tight_layout()

    viz_path = f'/content/{patient_id}_mri_viz.png'
    plt.savefig(viz_path, dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
    plt.close()

    # --- Build PDF ---
    from reportlab.lib.pagesizes import letter
    from reportlab.lib import colors
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
    from reportlab.lib.enums import TA_CENTER
    from datetime import datetime

    doc    = SimpleDocTemplate(output_path, pagesize=letter,
                               leftMargin=0.75*inch, rightMargin=0.75*inch,
                               topMargin=1*inch, bottomMargin=1*inch)
    styles = getSampleStyleSheet()
    story  = []

    title_style = ParagraphStyle('T', parent=styles['Heading1'],
                                  fontSize=22, alignment=TA_CENTER, spaceAfter=20)
    story.append(Paragraph("<b>MedLens Clinical Report</b>", title_style))
    story.append(Spacer(1, 0.2*inch))

    status_text = "YES — Clinically Significant" if sig_count > 0 else "NO — Benign"
    data = [
        ["Patient ID",           patient_id],
        ["Report Date",          datetime.now().strftime('%Y-%m-%d %H:%M')],
        ["PI-RADS Score",        f"{pirads} / 5"],
        ["Total Findings",       str(len(findings))],
        ["Clinically Significant", status_text],
    ]
    table = Table(data, colWidths=[2.2*inch, 4*inch])
    table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (0,-1), colors.HexColor('#eeeeee')),
        ('FONTNAME',   (0,0), (0,-1), 'Helvetica-Bold'),
        ('FONTSIZE',   (0,0), (-1,-1), 10),
        ('GRID',       (0,0), (-1,-1), 0.5, colors.grey),
    ]))
    story.append(table)
    story.append(Spacer(1, 0.3*inch))

    story.append(RLImage(viz_path, width=5.5*inch, height=2.5*inch))
    story.append(Spacer(1, 0.2*inch))

    story.append(Paragraph("<b>Clinical Findings</b>", styles['Heading2']))
    if findings:
        for f in findings:
            is_sig = f['significant']
            color_hex = '#cc0000' if is_sig else '#226622'
            label = "CLINICALLY SIGNIFICANT" if is_sig else "Benign"
            story.append(Paragraph(
                f"<font color='{color_hex}'><b>Finding {f['finding']}: {label}</b></font>"
                f" — Gleason: {f.get('gleason','N/A')}",
                styles['Normal']))
            story.append(Spacer(1, 0.1*inch))
    else:
        story.append(Paragraph("No lesions detected.", styles['Normal']))

    story.append(Spacer(1, 0.2*inch))
    story.append(Paragraph("<b>Clinical Recommendation</b>", styles['Heading2']))

    if sig_count > 0:
        rec = "Clinically significant findings detected. Recommend urological consultation and biopsy confirmation."
    elif pirads >= 3:
        rec = "Equivocal findings. Consider repeat MRI in 6–12 months or targeted biopsy if PSA rising."
    else:
        rec = "No significant findings. Continue routine surveillance with annual PSA."

    story.append(Paragraph(rec, styles['Normal']))
    story.append(Spacer(1, 0.3*inch))
    story.append(Paragraph(
        "<i>AI-assisted report. Requires radiologist confirmation. Not for clinical use without expert review.</i>",
        styles['Normal']))

    doc.build(story)
    print(f"✅ PDF saved: {output_path}")
    return output_path

# Test with 2 different patients
pdf1 = generate_pdf_report('ProstateX-0000')
pdf2 = generate_pdf_report('ProstateX-0009')
print("Both PDFs generated!")

In [ ]:
# Cell 17: Save all to Drive
import shutil, pickle

base = '/content/drive/MyDrive/MedLens'
import os
os.makedirs(base, exist_ok=True)

# Save indexes
shutil.copy('/content/patient_index.pkl',  f'{base}/patient_index.pkl')
shutil.copy('/content/patient_pirads.pkl', f'{base}/patient_pirads.pkl')

print("✅ Saved to Drive!")
print("\nAlso do: File → Save a copy in Drive")

In [ ]:
# Run in MedLens Colab notebook
import os
import shutil
import zipfile
import pickle

# Load patient index
with open('/content/patient_index.pkl', 'rb') as f:
    patient_index = pickle.load(f)

with open('/content/patient_pirads.pkl', 'rb') as f:
    patient_pirads = pickle.load(f)

# Pick 3 benign + 2 cancer patients
selected = ['ProstateX-0001', 'ProstateX-0009', 'ProstateX-0010',
            'ProstateX-0000', 'ProstateX-0004']

# Copy their DICOM files
export_dir = '/content/medlens_sample'
os.makedirs(export_dir, exist_ok=True)

for pid in selected:
    info = patient_index.get(pid)
    if not info:
        continue

    pid_dir = f'{export_dir}/{pid}'
    os.makedirs(pid_dir, exist_ok=True)

    # Copy T2W (first 5 slices only to save space)
    if info.get('t2'):
        t2_dst = f'{pid_dir}/t2'
        os.makedirs(t2_dst, exist_ok=True)
        files = sorted(os.listdir(info['t2'][0]))[:5]
        for f in files:
            shutil.copy(os.path.join(info['t2'][0], f), t2_dst)

    # Copy ADC (first 5 slices)
    if info.get('adc'):
        adc_dst = f'{pid_dir}/adc'
        os.makedirs(adc_dst, exist_ok=True)
        files = sorted(os.listdir(info['adc'][0]))[:5]
        for f in files:
            shutil.copy(os.path.join(info['adc'][0], f), adc_dst)

print(f"✅ Copied {len(selected)} patients")

# Zip it
with zipfile.ZipFile('/content/medlens_sample.zip', 'w') as z:
    for root, dirs, files in os.walk(export_dir):
        for f in files:
            z.write(os.path.join(root, f),
                    os.path.relpath(os.path.join(root, f), export_dir))

size = os.path.getsize('/content/medlens_sample.zip') / 1e6
print(f"✅ Zip created: {size:.1f} MB")

# Download it
from google.colab import files
files.download('/content/medlens_sample.zip')

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/MedLens'))


['patient_index.pkl', 'patient_pirads.pkl']
